# 03 — Bib identity resolution

Sweep `--ocr-every-n` (1, 5, 15) and plot identity-resolution rate vs OCR sample budget.
Per-track vote distributions show whether the failure mode is OCR misses (most reads return None) or genuine misreads (numbers but wrong).

In [ ]:
import sys, json, subprocess
from pathlib import Path
import matplotlib.pyplot as plt
sys.path.insert(0, '..')

VIDEO = Path('../data/raw_clips/match_1080p.mp4')
OUT = Path('../data/outputs')
PYTHON = sys.executable

In [ ]:
results = {}
for n in (1, 5, 15):
    out_dir = OUT / f'ocr_every_{n}'
    cmd = [PYTHON, '../scripts/run_pipeline.py',
           '--input', str(VIDEO),
           '--output', str(out_dir),
           '--start-frame', '4500',
           '--max-frames', '500',
           '--no-video',
           '--ocr-every-n', str(n)]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
    results[n] = json.loads((out_dir / 'match_1080p.json').read_text())['stats']
for n, s in results.items():
    print(f'ocr_every_n={n:2d}  resolved={s["identities_resolved"]:3d}  unresolved={s["identities_unresolved"]:3d}  ocr_total={s["ocr_samples_total"]}')

In [ ]:
ns = sorted(results.keys())
rate = [results[n]['identities_resolved'] / max(1, results[n]['identities_resolved']+results[n]['identities_unresolved']) for n in ns]
samples = [results[n]['ocr_samples_total'] for n in ns]
fig, ax1 = plt.subplots()
ax1.plot(ns, rate, 'o-', label='resolution rate'); ax1.set_xlabel('ocr_every_n'); ax1.set_ylabel('identity resolution rate')
ax2 = ax1.twinx(); ax2.plot(ns, samples, 's--', color='gray', label='OCR samples'); ax2.set_ylabel('OCR samples')
ax1.set_title('Identity resolution rate vs OCR sample budget'); plt.show()

## Per-track vote distribution

For each track, count how many OCR samples returned a number vs. None.
Tracks where the voter saw e.g. 10 reads and still couldn't reach `min_votes_for_identity` are usually colour failures — the bib HSV doesn't match the configured red/blue ranges.

In [ ]:
# Re-run with --persist-tracks to inspect raw OCR readings (requires extending the
# pickle to include identity_resolver._readings_by_track — not done in v1).